In [4]:
# ── MACRO ─────────────────────────────────────────────────────────────────────
ALGORITHM = "sparsify"   # <- change this to switch algorithm
# ──────────────────────────────────────────────────────────────────────────────

import subprocess
import sys
import json
import tempfile
from pathlib import Path

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODELS, PAPILO_PATH
from src.presolvers.papilo import _ALL_PAPILO_PRESOLVERS
from tools.parse_papilo_log import parse, to_metadata

STATIC_OUT = Path("/data/energy-system-preprocessing/presolve/static") / ALGORITHM
STATIC_OUT.mkdir(parents=True, exist_ok=True)

assert ALGORITHM in _ALL_PAPILO_PRESOLVERS, (
    f"Unknown algorithm '{ALGORITHM}'. "
    f"Valid options: {sorted(_ALL_PAPILO_PRESOLVERS)}"
)

print(f"Algorithm : {ALGORITHM}")
print(f"Output dir: {STATIC_OUT}")
print(f"Disabled  : {sorted(_ALL_PAPILO_PRESOLVERS - {ALGORITHM})}")

Algorithm : sparsify
Output dir: /data/energy-system-preprocessing/presolve/static/sparsify
Disabled  : ['cliquemerging', 'coefftightening', 'colsingleton', 'domcol', 'doubletoneq', 'dualfix', 'dualinfer', 'fixcontinuous', 'implint', 'parallelcols', 'parallelrows', 'probing', 'propagation', 'simpleprobing', 'simplifyineq', 'stuffing', 'substitution']


In [5]:
def run_static_presolve(model_dir: Path) -> bool:
    model_name = model_dir.name
    mps_in = model_dir / "original.mps"
    if not mps_in.exists():
        print(f"  SKIP: no original.mps in {model_dir}")
        return False

    out_dir = STATIC_OUT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    reduced   = out_dir / "reduced.mps"
    postsolve = out_dir / "reduced.postsolve"
    log_file  = out_dir / "output.txt"

    if reduced.exists():
        print(f"  skipped (already exists)", flush=True)
        return True

    # Disable every presolver except ALGORITHM via a temporary settings file.
    # PaPILO requires integer 0/1, not false/true.
    disabled = _ALL_PAPILO_PRESOLVERS - {ALGORITHM}
    settings_lines = [f"{p}.enabled = 0" for p in sorted(disabled)]

    with tempfile.NamedTemporaryFile(mode="w", suffix=".set", delete=False) as tf:
        tf.write("\n".join(settings_lines) + "\n")
        settings_path = Path(tf.name)

    cmd = [
        str(PAPILO_PATH), "presolve",
        "-f", str(mps_in),
        "-r", str(reduced),
        "-v", str(postsolve),
        "-p", str(settings_path),
        "--message.verbosity=4",
    ]

    try:
        with log_file.open("w") as log:
            process = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT)
    finally:
        settings_path.unlink(missing_ok=True)

    if process.returncode != 0 or not reduced.exists():
        print(f"  FAILED (rc={process.returncode})")
        return False

    rounds, model_path = parse(log_file.read_text().splitlines(keepends=True))
    meta = to_metadata(rounds, model_path)

    src_meta_path = model_dir / "metadata.json"
    existing: dict = json.loads(src_meta_path.read_text()) if src_meta_path.exists() else {}
    existing.update(meta)
    (out_dir / "metadata.json").write_text(json.dumps(existing, indent=2))

    return True

In [6]:
models = sorted(MODELS.iterdir())
print(f"{len(models)} models found in {MODELS}")

failed = []

for i, model_dir in enumerate(models, 1):
    print(f"[{i}/{len(models)}] {model_dir.name}", flush=True)
    ok = run_static_presolve(model_dir)
    if not ok:
        failed.append(model_dir.name)
    print(flush=True)

if failed:
    print(f"\n{len(failed)} model(s) failed: {failed}")
else:
    print(f"\nAll {len(models)} models presolved successfully.")

445 models found in /data/energy-system-preprocessing/models
[1/445] r10_res168_f0.0000_t0.0192

[2/445] r10_res168_f0.0000_t0.0833

[3/445] r10_res168_f0.0000_t1.0000

[4/445] r10_res168_f0.2500_t0.5000

[5/445] r10_res168_f0.5000_t1.0000

[6/445] r10_res1_f0.0000_t0.0192

[7/445] r10_res1_f0.0000_t0.0833

[8/445] r10_res1_f0.0000_t1.0000

[9/445] r10_res1_f0.2500_t0.5000

[10/445] r10_res1_f0.5000_t1.0000

[11/445] r10_res24_f0.0000_t0.0192

[12/445] r10_res24_f0.0000_t0.0833

[13/445] r10_res24_f0.0000_t1.0000

[14/445] r10_res24_f0.2500_t0.5000

[15/445] r10_res24_f0.5000_t1.0000

[16/445] r10_res8_f0.0000_t0.0192

[17/445] r10_res8_f0.0000_t0.0833

[18/445] r10_res8_f0.0000_t1.0000

[19/445] r10_res8_f0.2500_t0.5000

[20/445] r10_res8_f0.5000_t1.0000

[21/445] r11_res168_f0.0000_t0.0192

[22/445] r11_res168_f0.0000_t0.0833

[23/445] r11_res168_f0.0000_t1.0000

[24/445] r11_res168_f0.2500_t0.5000

[25/445] r11_res168_f0.5000_t1.0000

[26/445] r11_res1_f0.0000_t0.0192

[27/445] r11_